In [1]:
import numpy as np

In [2]:
import numpy as np

def butterfly(a, b, twiddle):
    """
    a, b: Liczby zespolone (wejścia)
    twiddle: Liczba zespolona (współczynnik W_N^k)
    """
    # Operacja motylkowa:
    # A = a + b*W
    # B = a - b*W
    temp = b * twiddle
    return a + temp, a - temp

def fft_radix2(x):
    N = len(x)
    if N <= 1: return x
    
    # Podział na parzyste i nieparzyste
    even = fft_radix2(x[0::2])
    odd = fft_radix2(x[1::2])
    
    # Współczynniki Twiddle
    T = [np.exp(-2j * np.pi * k / N) for k in range(N // 2)]
    
    # Łączenie (Butterfly)
    return [even[k] + T[k] * odd[k] for k in range(N // 2)] + \
           [even[k] - T[k] * odd[k] for k in range(N // 2)]

In [51]:
def fixed_point_butterfly(A_re, A_im, B_re, B_im, W_re, W_im):
    """
    Wszystkie argumenty wejściowe muszą być liczbami całkowitymi (int)
    reprezentującymi ułamki w formacie Q1.15.
    """
    
    # ----------------------------------------------------
    # KROK 1: Mnożenie zespolone (B * W)
    # W FPGA mnożenie dwóch liczb 16-bitowych daje wynik 32-bitowy (format Q2.30)
    # ----------------------------------------------------
    temp_re = (B_re * W_re) - (B_im * W_im)
    temp_im = (B_re * W_im) + (B_im * W_re)
    
    # ----------------------------------------------------
    # KROK 2: Skalowanie (Bit-shift)
    # Przesunięcie bitowe o 15 w prawo ucina "nadmiarowe" bity ułamkowe 
    # i przywraca wynik mnożenia do bazowego formatu Q1.15
    # ----------------------------------------------------
    temp_re = temp_re >> 15
    temp_im = temp_im >> 15
    
    # ----------------------------------------------------
    # KROK 3: Operacja sumowania i odejmowania (Butterfly)
    # ----------------------------------------------------
    out_A_re = A_re + temp_re
    out_A_im = A_im + temp_im
    
    out_B_re = A_re - temp_re
    out_B_im = A_im - temp_im
    
    # W rygorystycznym modelu hardware'owym tutaj następuje ew. obcięcie do 16-bit
    # np. ograniczenie wyników do zakresu [-32768, 32767] (Saturacja)
    
    return out_A_re, out_A_im, out_B_re, out_B_im

In [52]:
# Funkcje pomocnicze do konwersji
def float_to_q15(val):
    # Skalowanie i obcięcie do zakresu int16
    return int(np.clip(val * 32768.0, -32768, 32767))

def q15_to_float(val):
    return val / 32768.0

# 1. Dane wejściowe w float (tak jak czytasz z mikrofonu)
a_re_f, a_im_f = 0.5, 0.1
b_re_f, b_im_f = -0.3, 0.4
w_re_f, w_im_f = 0.707, -0.707 # Zbliżenie np. exp(-j*pi/4)

# 2. Konwersja na liczby całkowite dla FPGA
A_re = float_to_q15(a_re_f)
A_im = float_to_q15(a_im_f)
B_re = float_to_q15(b_re_f)
B_im = float_to_q15(b_im_f)
W_re = float_to_q15(w_re_f)
W_im = float_to_q15(w_im_f)

# 3. Wykonanie symulacji sprzętowej (tylko int i przesunięcia!)
out_A_re, out_A_im, out_B_re, out_B_im = fixed_point_butterfly(A_re, A_im, B_re, B_im, W_re, W_im)

# 4. Porównanie z "idealną" matematyką zmiennoprzecinkową
A_ideal = complex(a_re_f, a_im_f) + complex(b_re_f, b_im_f) * complex(w_re_f, w_im_f)
B_ideal = complex(a_re_f, a_im_f) - complex(b_re_f, b_im_f) * complex(w_re_f, w_im_f)

print("--- Wynik Hardware (Q1.15 -> Float) ---")
print(f"A' = {q15_to_float(out_A_re):.4f} + {q15_to_float(out_A_im):.4f}j")
print(f"B' = {q15_to_float(out_B_re):.4f} + {q15_to_float(out_B_im):.4f}j")

print("\n--- Wynik Idealny (Float64) ---")
print(f"A' = {A_ideal.real:.4f} + {A_ideal.imag:.4f}j")
print(f"B' = {B_ideal.real:.4f} + {B_ideal.imag:.4f}j")

--- Wynik Hardware (Q1.15 -> Float) ---
A' = 0.5707 + 0.5948j
B' = 0.4293 + -0.3949j

--- Wynik Idealny (Float64) ---
A' = 0.5707 + 0.5949j
B' = 0.4293 + -0.3949j
